<a href="https://colab.research.google.com/github/epkalibbala/Bar-code-scanner/blob/main/AlgoTrading_one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fredapi
!pip install pmdarima
!pip install oandapyV20

In [ ]:
"""
ULTIMATE GBP/USD AUTOARIMA MEAN REVERSION STRATEGY
COMPLETE DEBUGGED VERSION - PRESERVING ALL OANDA LOGIC
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fredapi import Fred
from pmdarima import auto_arima
from scipy.optimize import minimize_scalar
from scipy import stats, signal
from scipy.signal import savgol_filter, find_peaks
import warnings
warnings.filterwarnings('ignore')
from oandapyV20 import API
from oandapyV20.endpoints.instruments import InstrumentsCandles
from datetime import datetime, timedelta
import traceback

# =============================================================================
# 1. COMPLETE DATA LOADING - FIXED OANDA INTEGRATION
# =============================================================================

def load_gbp_data_with_regimes(fred_api_key, oanda_token):
    """FIXED: Complete OANDA + FRED data loading with error handling"""

    # ----------------------------
    # 1. OANDA DATA LOADING - FIXED
    # ----------------------------
    print("🔄 Loading GBP/USD OHLC from OANDA...")

    try:
        # Initialize OANDA API
        oanda_api = API(access_token=oanda_token, environment="practice")  # Use "live" for real account

        # Proper date formatting for OANDA
        params = {
            "granularity": "D",
            "from": "2014-01-01T00:00:00Z",
            "to": "2026-01-21T00:00:00Z",
            "price": "M"  # Midpoint candles
        }

        # Make API request
        r = InstrumentsCandles(instrument="GBP_USD", params=params)
        oanda_api.request(r)

        # Parse response
        if 'candles' not in r.response:
            raise ValueError("No 'candles' in OANDA response")

        candles = r.response["candles"]
        print(f"✅ Received {len(candles)} candles from OANDA")

        # Parse OHLC data
        ohlc_data = []
        for candle in candles:
            if candle['complete']:
                try:
                    ohlc_data.append({
                        'time': candle['time'],
                        'GBP_USD': float(candle['mid']['c']),
                        'GBP_USD_HIGH': float(candle['mid']['h']),
                        'GBP_USD_LOW': float(candle['mid']['l']),
                        'GBP_USD_OPEN': float(candle['mid']['o']),
                        'GBP_USD_VOLUME': int(candle['volume'])
                    })
                except KeyError as e:
                    print(f"⚠ Missing key in candle data: {e}")
                    continue

        if not ohlc_data:
            raise ValueError("No complete candles found in OANDA data")

        # Create DataFrame
        df_ohlc = pd.DataFrame(ohlc_data)
        df_ohlc['time'] = pd.to_datetime(df_ohlc['time'], utc=True)
        df_ohlc.set_index('time', inplace=True)

        print(f"✅ Processed {len(df_ohlc)} OHLC records")
        print(f"   Date range: {df_ohlc.index.min()} to {df_ohlc.index.max()}")
        print(f"   GBP/USD range: {df_ohlc['GBP_USD'].min():.4f} - {df_ohlc['GBP_USD'].max():.4f}")

    except Exception as e:
        print(f"❌ OANDA API ERROR: {e}")
        print("Troubleshooting:")
        print("1. Check OANDA token is valid")
        print("2. Check instrument 'GBP_USD' is available in your account")
        print("3. Try smaller date range first")
        traceback.print_exc()
        return None

    # ----------------------------
    # 2. FRED MACRO DATA - FIXED
    # ----------------------------
    print("\n🔄 Loading macro data from FRED...")

    try:
        fred = Fred(api_key=fred_api_key)

        # Define FRED series with multiple fallback options
        series_to_fetch = {
            'UK_PROD': ['CPMNACSCAB1GQGB', 'GBRPROCONAISMEI', 'GBRPROINDQISMEI'],
            'US_PROD': ['OPHNFB', 'OPHMFG', 'IPB50001N'],
            'UK_RATE': ['IUDSOIA', 'IR3TIB01GBM156N', 'GBRRUKMIXDEM'],
            'US_RATE': ['FEDFUNDS', 'DFF', 'FF'],
            'UK_INFLATION': ['GBRCPIALLQINMEI', 'GBRCPICORMINMEI', 'CPALTT01GBM657N'],
            'US_INFLATION': ['CPIAUCSL', 'CPILFESL', 'CPALTT01USM657N'],
            'VIX': ['VIXCLS', 'VXOCLS']
        }

        # Fetch data with fallbacks
        macro_data = {}
        for name, series_ids in series_to_fetch.items():
            data = None
            for series_id in series_ids:
                try:
                    series = fred.get_series(series_id)
                    if series is not None and len(series) > 10:
                        data = series
                        print(f"  ✓ {name}: {series_id} ({len(series)} points)")
                        break
                except Exception as e:
                    continue

            if data is not None:
                macro_data[name] = data
            else:
                print(f"  ⚠ {name}: No data available")

        # Create macro DataFrame
        if macro_data:
            df_macro = pd.DataFrame(macro_data)
            # df_macro.index = pd.to_datetime(df_macro.index)
            df_macro.index = pd.to_datetime(df_macro.index).tz_localize("UTC")
            print(f"✅ Loaded {len(df_macro.columns)} macro series")
        else:
            print("⚠ No FRED data loaded - creating empty DataFrame")
            df_macro = pd.DataFrame(index=df_ohlc.index)

    except Exception as e:
        print(f"❌ FRED API ERROR: {e}")
        print("Creating placeholder macro data...")
        df_macro = pd.DataFrame(index=df_ohlc.index)

    # ----------------------------
    # 3. DATA MERGING - FIXED
    # ----------------------------
    print("\n🔄 Merging OANDA + FRED data...")

    try:
        # Resample macro data to daily frequency
        if not df_macro.empty:
            df_macro_daily = df_macro.resample('D').ffill()
        else:
            df_macro_daily = pd.DataFrame(index=df_ohlc.index)

        # Merge data
        df = df_ohlc.join(df_macro_daily, how='left')

        # Fill missing macro data (limited forward fill)
        for col in df.columns:
            if col not in ['GBP_USD', 'GBP_USD_HIGH', 'GBP_USD_LOW', 'GBP_USD_OPEN', 'GBP_USD_VOLUME']:
                df[col] = df[col].ffill(limit=10)

        # Drop rows with missing GBP/USD price
        df = df.dropna(subset=['GBP_USD'])

        print(f"✅ Final dataset shape: {df.shape}")
        print(f"   Columns: {list(df.columns)}")

    except Exception as e:
        print(f"❌ Data merging error: {e}")
        traceback.print_exc()
        return None

    # ----------------------------
    # 4. FEATURE ENGINEERING - COMPLETE
    # ----------------------------
    print("\n🔄 Applying feature engineering...")

    try:
        # Add missing functions that were called but not defined
        df = apply_all_features(df)

        print(f"✅ Feature engineering complete")
        print(f"   Final data shape: {df.shape}")

        return df

    except Exception as e:
        print(f"❌ Feature engineering error: {e}")
        traceback.print_exc()
        return None

# =============================================================================
# 2. COMPLETE FEATURE ENGINEERING FUNCTIONS
# =============================================================================

def detect_gbp_regimes(gbp_series):
    """Detect major GBP regimes - COMPLETE IMPLEMENTATION"""
    regimes = pd.Series('normal', index=gbp_series.index)

    # Ensure index is UTC
    idx = gbp_series.index
    if idx.tz is None:
        idx = idx.tz_localize("UTC")
    else:
        idx = idx.tz_convert("UTC")

    # Define major event periods
    major_events = {
        '2008_crisis': ('2008-09-15', '2009-03-31'),
        'brexit_vote': ('2016-06-23', '2016-10-31'),
        'brexit_implementation': ('2020-01-31', '2020-12-31'),
        'covid_crash': ('2020-03-01', '2020-06-30'),
        'truss_mini_budget': ('2022-09-23', '2022-10-20'),
        'normal': ('2014-01-01', '2026-01-21')  # Default
    }

    for event, (start, end) in major_events.items():
        start_ts = pd.Timestamp(start, tz="UTC")
        end_ts   = pd.Timestamp(end, tz="UTC")

        mask = (idx >= start_ts) & (idx <= end_ts)
        regimes.loc[mask] = event

        # mask = (gbp_series.index >= pd.Timestamp(start)) & (gbp_series.index <= pd.Timestamp(end))
        # regimes[mask] = event

    return regimes

def classify_prod_regime_enhanced(prod_diff, prod_momentum, window=504):
    """Enhanced regime classification - COMPLETE"""
    if prod_diff.empty:
        return pd.Series('NP', index=prod_diff.index)

    regimes = pd.Series('NP', index=prod_diff.index)

    for i in range(window, len(prod_diff)):
        if pd.isna(prod_diff.iloc[i]):
            continue

        window_data = prod_diff.iloc[i-window:i].dropna()
        if len(window_data) < 100:
            continue

        try:
            # Dynamic thresholds
            q25 = np.percentile(window_data, 25)
            q75 = np.percentile(window_data, 75)

            current_val = prod_diff.iloc[i]
            current_mom = prod_momentum.iloc[i] if not pd.isna(prod_momentum.iloc[i]) else 0

            if current_val <= q25 and current_mom < 0:
                regimes.iloc[i] = 'LP_strong'
            elif current_val <= q25:
                regimes.iloc[i] = 'LP'
            elif current_val >= q75 and current_mom > 0:
                regimes.iloc[i] = 'HP_strong'
            elif current_val >= q75:
                regimes.iloc[i] = 'HP'
            elif abs(current_mom) > np.std(window_data):
                regimes.iloc[i] = 'NP_momentum'
            else:
                regimes.iloc[i] = 'NP'
        except:
            regimes.iloc[i] = 'NP'

    return regimes.ffill().bfill()

def apply_all_features(df):
    """Apply ALL feature engineering steps - COMPLETE"""

    # 1. Basic price features
    df['GBP_RETURNS'] = df['GBP_USD'].pct_change()
    df['GBP_LOG_RETURNS'] = np.log(df['GBP_USD'] / df['GBP_USD'].shift(1))

    # 2. Volatility features (using OANDA high/low)
    if 'GBP_USD_HIGH' in df.columns and 'GBP_USD_LOW' in df.columns:
        df['GBP_DAILY_RANGE'] = (df['GBP_USD_HIGH'] - df['GBP_USD_LOW']) / df['GBP_USD']
        df['GBP_DAILY_RANGE_PCT'] = df['GBP_DAILY_RANGE'] * 100

        # True Range
        tr1 = df['GBP_USD_HIGH'] - df['GBP_USD_LOW']
        tr2 = abs(df['GBP_USD_HIGH'] - df['GBP_USD'].shift(1))
        tr3 = abs(df['GBP_USD_LOW'] - df['GBP_USD'].shift(1))
        df['TRUE_RANGE'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        df['ATR_14'] = df['TRUE_RANGE'].rolling(14).mean()

    # 3. Volatility measures
    df['GBP_VOLATILITY_20D'] = df['GBP_RETURNS'].rolling(20).std() * np.sqrt(252)
    df['GBP_VOLATILITY_60D'] = df['GBP_RETURNS'].rolling(60).std() * np.sqrt(252)

    # 4. Regime detection
    df['GBP_MAJOR_REGIME'] = detect_gbp_regimes(df['GBP_USD'])

    # 5. Productivity features (if data available)
    if all(col in df.columns for col in ['UK_PROD', 'US_PROD']):
        df['PROD_DIFF'] = df['UK_PROD'] - df['US_PROD']
        df['PROD_MOMENTUM'] = df['PROD_DIFF'].diff(63)
        df['PROD_REGIME'] = classify_prod_regime_enhanced(df['PROD_DIFF'], df['PROD_MOMENTUM'])

    # 6. Rate features
    if all(col in df.columns for col in ['UK_RATE', 'US_RATE']):
        df['RATE_DIFF'] = df['UK_RATE'] - df['US_RATE']
        df['RATE_EXPECTATION'] = df['RATE_DIFF'].diff(21)

        # Create rate regime
        bins = [-np.inf, -0.25, 0.25, np.inf]
        labels = ['bearish', 'neutral', 'bullish']
        df['RATE_REGIME'] = pd.cut(df['RATE_EXPECTATION'].fillna(0), bins=bins, labels=labels)

    # 7. Inflation features
    if all(col in df.columns for col in ['UK_INFLATION', 'US_INFLATION']):
        df['INFLATION_DIFF'] = df['UK_INFLATION'] - df['US_INFLATION']

    # 8. VIX features
    if 'VIX' in df.columns:
        df['VIX_SMA_252'] = df['VIX'].rolling(252).mean()
        df['VIX_ADJUSTED_VOL'] = df['GBP_VOLATILITY_20D'] * (df['VIX'] / df['VIX_SMA_252'])

    # 9. Combined regime
    df['COMBINED_REGIME'] = calculate_combined_regime(df)

    # 10. Technical indicators
    df['SMA_20'] = df['GBP_USD'].rolling(20).mean()
    df['SMA_50'] = df['GBP_USD'].rolling(50).mean()
    df['SMA_200'] = df['GBP_USD'].rolling(200).mean()

    df['RSI_14'] = calculate_rsi(df['GBP_USD'], 14)

    # Fill NaN values
    df = df.ffill(limit=10).bfill(limit=10)

    return df

def calculate_rsi(prices, period=14):
    """Calculate RSI indicator"""
    deltas = prices.diff()
    gain = (deltas.where(deltas > 0, 0)).rolling(window=period).mean()
    loss = (-deltas.where(deltas < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def calculate_combined_regime(df):
    """Calculate combined regime score from multiple indicators"""
    score = pd.Series(0, index=df.index)

    # Productivity regime score
    if 'PROD_REGIME' in df.columns:
        prod_map = {'LP_strong': -2, 'LP': -1, 'NP': 0, 'NP_momentum': 0, 'HP': 1, 'HP_strong': 2}
        prod_score = df['PROD_REGIME'].map(prod_map).fillna(0)
        score += prod_score

    # Rate regime score
    if 'RATE_REGIME' in df.columns:
        rate_map = {'bearish': -1, 'neutral': 0, 'bullish': 1}
        # rate_score = df['RATE_REGIME'].map(rate_map).fillna(0)
        # score += rate_score
        rate_score = (
        df['RATE_REGIME']
        .map(rate_map)
        .astype(float)   # 🔑 FORCE numeric
        .fillna(0.0)
        )

        score = score + rate_score


    # VIX adjustment
    if 'VIX_ADJUSTED_VOL' in df.columns:
        try:
            vix_quantile = df['VIX_ADJUSTED_VOL'].rolling(252).quantile(0.75)
            high_vol = df['VIX_ADJUSTED_VOL'] > vix_quantile
            score[high_vol] = 0
        except:
            pass

    # Major event override
    if 'GBP_MAJOR_REGIME' in df.columns:
        major_event = df['GBP_MAJOR_REGIME'].isin(['brexit_vote', 'covid_crash', 'truss_mini_budget'])
        score[major_event] = 0

    # Create final regime categories
    bins = [-np.inf, -1.5, -0.5, 0.5, 1.5, np.inf]
    labels = ['strong_bearish', 'bearish', 'neutral', 'bullish', 'strong_bullish']

    return pd.cut(score, bins=bins, labels=labels)

# =============================================================================
# 3. POSITION SIZING - COMPLETE IMPLEMENTATION
# =============================================================================

class GBPPositionSizer:
    """Complete position sizing implementation"""

    def __init__(self, capital, base_risk=0.01, max_position_pct=0.08):
        self.capital = capital
        self.base_risk = base_risk
        self.max_position_pct = max_position_pct
        self.position_history = []

    def calculate_var(self, returns, confidence=0.95):
        """Calculate Value at Risk"""
        if len(returns) < 20:
            return 0.02

        returns_clean = returns[~np.isnan(returns)]
        if len(returns_clean) < 10:
            return 0.02

        sorted_returns = np.sort(returns_clean)
        idx = int((1 - confidence) * len(sorted_returns))
        idx = max(0, min(idx, len(sorted_returns) - 1))

        return abs(sorted_returns[idx])

    def calculate_units(self, entry_price, stop_loss, recent_returns, regime, volatility):
        """Calculate position size with all constraints"""

        # Basic validation
        if entry_price <= 0 or stop_loss <= 0:
            return 0

        # 1. Base risk calculation
        risk_amount = self.capital * self.base_risk
        risk_per_unit = abs(entry_price - stop_loss)

        if risk_per_unit == 0:
            return 0

        base_units = risk_amount / risk_per_unit

        # 2. Volatility adjustment
        if volatility > 0.015:
            vol_mult = 0.7
        elif volatility < 0.008:
            vol_mult = 1.3
        else:
            vol_mult = 1.0

        base_units *= vol_mult

        # 3. Regime adjustment
        regime_mult = {
            'strong_bullish': 1.2,
            'bullish': 1.1,
            'neutral': 1.0,
            'bearish': 0.9,
            'strong_bearish': 0.8,
            'crisis': 0.5,
            'brexit': 0.3
        }.get(regime, 1.0)

        base_units *= regime_mult

        # 4. VaR constraint
        if len(recent_returns) > 10:
            var = self.calculate_var(recent_returns)
            if var > 0:
                var_max = (self.capital * 0.02) / (entry_price * var)
                base_units = min(base_units, var_max)

        # 5. Maximum position constraint
        max_units = (self.capital * self.max_position_pct) / entry_price
        base_units = min(base_units, max_units)

        # 6. Round to mini lots (10,000 units)
        standard_lot = 100000  # GBP standard lot
        mini_lot = 10000      # GBP mini lot

        units = int(base_units / mini_lot) * mini_lot

        # Ensure minimum trade size
        if units < mini_lot:
            return 0

        return units

# =============================================================================
# 4. AUTOARIMA FORECASTING - COMPLETE
# =============================================================================

class GBPAutoARIMA:
    """Complete AutoARIMA implementation"""

    def __init__(self, window=60, min_window=20):
        self.window = window
        self.min_window = min_window
        self.model_cache = {}
        self.forecast_cache = {}

    def forecast_with_volatility_adaptation(self, prices, volatility, regime):
        """Generate forecast with volatility adaptation"""

        if len(prices) < self.min_window:
            return np.mean(prices[-10:]) if len(prices) >= 10 else prices[-1], 10

        # Adjust window based on volatility
        if volatility > 0.015:
            effective_window = min(30, len(prices))
        elif volatility < 0.008:
            effective_window = min(90, len(prices))
        else:
            effective_window = min(self.window, len(prices))

        effective_window = max(self.min_window, effective_window)

        # Cache key
        cache_key = f"{regime}_{effective_window}"

        try:
            if cache_key in self.model_cache:
                # Update existing model
                model = self.model_cache[cache_key]
                model.update(prices[-effective_window:])
            else:
                # Build new model
                model = auto_arima(
                    prices[-effective_window:],
                    seasonal=False,
                    max_p=3,
                    max_q=3,
                    max_d=1,
                    suppress_warnings=True,
                    error_action='ignore',
                    stepwise=True,
                    n_jobs=-1,
                    trace=False,
                    information_criterion='aic'
                )
                self.model_cache[cache_key] = model

            # Generate forecast
            forecast = model.predict(n_periods=1)[0]

            # Apply regime adjustment
            if regime in ['strong_bullish', 'bullish']:
                forecast = forecast * 1.001
            elif regime in ['strong_bearish', 'bearish']:
                forecast = forecast * 0.999

            return forecast, effective_window

        except Exception as e:
            # Fallback to simple average
            return np.mean(prices[-effective_window:]), effective_window

# =============================================================================
# 5. MAIN STRATEGY CLASS - COMPLETE
# =============================================================================

class UltimateGBPStrategy:
    """Complete GBP/USD mean reversion strategy"""

    def __init__(self, initial_capital=100000):
        self.initial_capital = initial_capital
        self.capital = initial_capital
        self.position_sizer = GBPPositionSizer(initial_capital)
        self.arima_model = GBPAutoARIMA(window=60)

        # Trading state
        self.position = 0
        self.position_direction = 0  # 1 for long, -1 for short
        self.entry_price = 0.0
        self.entry_stop = 0.0
        self.entry_tp = 0.0
        self.entry_date = None
        self.entry_regime = None

        # Performance tracking
        self.equity_curve = [initial_capital]
        self.trades = []
        self.regime_history = []

        # Strategy parameters
        self.params = {
            'base_atr_mult': 0.6,
            'sl_mult': 2.8,
            'tp_mult': 1.5,
            'spread_pips': 3,
            'min_signal_quality': 0.3,
            'max_trade_duration': 20  # days
        }

    def calculate_signal_quality(self, deviation, atr, volatility, regime):
        """Calculate signal quality score"""
        if atr <= 0:
            return 0.0

        quality = 1.0

        # 1. Deviation strength
        deviation_strength = abs(deviation) / (atr + 1e-6)
        quality *= min(deviation_strength / 2.0, 1.0)

        # 2. Volatility adjustment
        if volatility > 0.02:
            quality *= 0.5
        elif volatility < 0.005:
            quality *= 0.7

        # 3. Regime quality
        regime_quality = {
            'neutral': 1.0,
            'bullish': 0.9,
            'bearish': 0.9,
            'strong_bullish': 0.8,
            'strong_bearish': 0.8,
        }.get(regime, 0.5)
        quality *= regime_quality

        return max(0.0, min(1.0, quality))

    def run_backtest(self, df, verbose=True):
        """Run complete backtest"""

        if verbose:
            print(f"🏴󠁧󠁢󠁥󠁮󠁧󠁿 Running GBP strategy on {len(df)} periods...")

        # Ensure required columns exist
        required_cols = ['GBP_USD', 'GBP_RETURNS']
        for col in required_cols:
            if col not in df.columns:
                print(f"❌ Missing required column: {col}")
                return self

        returns = df['GBP_RETURNS'].fillna(0).values

        for i in range(100, len(df)):  # Start after warmup
            current_date = df.index[i]
            current_price = df['GBP_USD'].iloc[i]

            # Skip if NaN price
            if pd.isna(current_price):
                self.equity_curve.append(self.equity_curve[-1])
                continue

            # Get current regime
            current_regime = 'neutral'
            if 'COMBINED_REGIME' in df.columns:
                regime_val = df['COMBINED_REGIME'].iloc[i]
                if not pd.isna(regime_val):
                    current_regime = str(regime_val)

            self.regime_history.append(current_regime)

            # Get volatility
            current_vol = 0.01
            if 'GBP_VOLATILITY_20D' in df.columns:
                vol_val = df['GBP_VOLATILITY_20D'].iloc[i]
                if not pd.isna(vol_val):
                    current_vol = vol_val

            # Generate forecast
            prices = df['GBP_USD'].iloc[:i+1].values
            forecast, window_used = self.arima_model.forecast_with_volatility_adaptation(
                prices, current_vol, current_regime
            )

            # Calculate ATR
            atr_value = 0.001
            if 'ATR_14' in df.columns:
                atr_val = df['ATR_14'].iloc[i]
                if not pd.isna(atr_val):
                    atr_value = atr_val

            # Calculate dynamic thresholds
            entry_threshold = atr_value * self.params['base_atr_mult']
            stop_threshold = atr_value * self.params['sl_mult']

            # Volatility adjustments
            if current_vol > 0.015:
                entry_threshold *= 1.2
                stop_threshold *= 1.3

            # Calculate deviation from forecast
            deviation = current_price - forecast

            # Calculate signal quality
            signal_quality = self.calculate_signal_quality(
                deviation, atr_value, current_vol, current_regime
            )

            # Generate trading signal
            signal = 0
            if signal_quality > self.params['min_signal_quality']:
                if deviation < -entry_threshold:
                    signal = 1  # Buy signal (GBP undervalued)
                elif deviation > entry_threshold:
                    signal = -1  # Sell signal (GBP overvalued)

            # ENTER NEW TRADE
            if signal != 0 and self.position == 0:
                # Calculate stop loss and take profit
                if signal == 1:  # Long
                    stop_loss = current_price - stop_threshold
                    take_profit = current_price + (stop_threshold * self.params['tp_mult'])
                else:  # Short
                    stop_loss = current_price + stop_threshold
                    take_profit = current_price - (stop_threshold * self.params['tp_mult'])

                # Calculate position size
                recent_returns = returns[max(0, i-50):i]
                units = self.position_sizer.calculate_units(
                    current_price, stop_loss, recent_returns,
                    current_regime, current_vol
                )

                # Execute trade if size is valid
                if units > 0:
                    self.position = units if signal == 1 else -units
                    self.position_direction = signal
                    self.entry_price = current_price
                    self.entry_stop = stop_loss
                    self.entry_tp = take_profit
                    self.entry_date = current_date
                    self.entry_regime = current_regime

                    # Record trade entry
                    self.trades.append({
                        'entry_date': current_date,
                        'entry_price': current_price,
                        'stop_loss': stop_loss,
                        'take_profit': take_profit,
                        'direction': 'long' if signal == 1 else 'short',
                        'units': abs(units),
                        'signal_quality': signal_quality,
                        'regime': current_regime,
                        'position_value': abs(units * current_price)
                    })

            # MANAGE EXISTING POSITION
            elif self.position != 0:
                exit_trade = False
                exit_reason = ""
                current_pnl = 0

                if self.position > 0:  # Long position
                    current_pnl = (current_price - self.entry_price) * self.position

                    # Check exit conditions
                    if current_price <= self.entry_stop:
                        exit_trade = True
                        exit_reason = "stop_loss"
                    elif current_price >= self.entry_tp:
                        exit_trade = True
                        exit_reason = "take_profit"
                    elif current_price >= forecast:
                        exit_trade = True
                        exit_reason = "forecast_reached"
                    elif (current_date - self.entry_date).days > self.params['max_trade_duration']:
                        exit_trade = True
                        exit_reason = "time_limit"

                else:  # Short position
                    current_pnl = (self.entry_price - current_price) * abs(self.position)

                    if current_price >= self.entry_stop:
                        exit_trade = True
                        exit_reason = "stop_loss"
                    elif current_price <= self.entry_tp:
                        exit_trade = True
                        exit_reason = "take_profit"
                    elif current_price <= forecast:
                        exit_trade = True
                        exit_reason = "forecast_reached"
                    elif (current_date - self.entry_date).days > self.params['max_trade_duration']:
                        exit_trade = True
                        exit_reason = "time_limit"

                # Exit trade if conditions met
                if exit_trade:
                    # Calculate costs
                    spread_cost = self.params['spread_pips'] * 0.0001 * abs(self.position)
                    commission = max(5.0, abs(current_pnl) * 0.0002)
                    net_pnl = current_pnl - spread_cost - commission

                    # Update capital
                    self.capital += net_pnl

                    # Update trade record
                    if self.trades:
                        last_trade = self.trades[-1]
                        last_trade.update({
                            'exit_date': current_date,
                            'exit_price': current_price,
                            'exit_reason': exit_reason,
                            'net_pnl': net_pnl,
                            'return_pct': (net_pnl / (self.entry_price * abs(self.position))) * 100,
                            'duration_days': (current_date - last_trade['entry_date']).days
                        })

                    # Reset position
                    self.position = 0
                    self.position_direction = 0
                    self.entry_price = 0.0

            # Update equity curve
            if self.position != 0:
                # Calculate unrealized P&L
                if self.position > 0:
                    unrealized_pnl = (current_price - self.entry_price) * self.position
                else:
                    unrealized_pnl = (self.entry_price - current_price) * abs(self.position)

                current_equity = self.capital + unrealized_pnl
            else:
                current_equity = self.capital

            self.equity_curve.append(current_equity)

        if verbose:
            print(f"✅ Backtest complete")
            print(f"   Total trades: {len(self.trades)}")
            print(f"   Final capital: ${self.capital:,.2f}")
            print(f"   Total return: {((self.capital/self.initial_capital)-1)*100:.2f}%")

        return self

# =============================================================================
# 6. PERFORMANCE ANALYTICS - COMPLETE
# =============================================================================

class GBPPerformanceAnalytics:
    """Complete performance analytics"""

    def __init__(self, strategy):
        self.strategy = strategy

        # Create trades DataFrame
        if strategy.trades:
            self.trades_df = pd.DataFrame(strategy.trades)
        else:
            self.trades_df = pd.DataFrame()

    def calculate_metrics(self):
        """Calculate all performance metrics"""
        metrics = {}

        if self.trades_df.empty:
            metrics['total_trades'] = 0
            metrics['win_rate'] = 0
            metrics['sharpe_ratio'] = 0
            metrics['max_drawdown'] = 0
            return metrics

        # Basic metrics
        metrics['total_trades'] = len(self.trades_df)
        metrics['winning_trades'] = len(self.trades_df[self.trades_df['net_pnl'] > 0])
        metrics['losing_trades'] = len(self.trades_df[self.trades_df['net_pnl'] < 0])
        metrics['win_rate'] = metrics['winning_trades'] / metrics['total_trades'] if metrics['total_trades'] > 0 else 0

        # Return metrics
        total_return_pct = ((self.strategy.capital / self.strategy.initial_capital) - 1) * 100
        metrics['total_return_pct'] = total_return_pct

        # Sharpe ratio
        if 'return_pct' in self.trades_df.columns:
            returns = self.trades_df['return_pct'].dropna().values / 100
            if len(returns) > 1:
                sharpe = np.mean(returns) / (np.std(returns) + 1e-6) * np.sqrt(252)
                metrics['sharpe_ratio'] = sharpe
            else:
                metrics['sharpe_ratio'] = 0

        # Drawdown
        equity = np.array(self.strategy.equity_curve)
        if len(equity) > 1:
            peak = np.maximum.accumulate(equity)
            drawdown = (equity - peak) / peak
            metrics['max_drawdown_pct'] = np.min(drawdown) * 100 if len(drawdown) > 0 else 0

        # Profit factor
        if 'net_pnl' in self.trades_df.columns:
            gross_profit = self.trades_df[self.trades_df['net_pnl'] > 0]['net_pnl'].sum()
            gross_loss = abs(self.trades_df[self.trades_df['net_pnl'] < 0]['net_pnl'].sum())
            metrics['profit_factor'] = gross_profit / gross_loss if gross_loss > 0 else np.inf

        # Average trade metrics
        if 'return_pct' in self.trades_df.columns:
            metrics['avg_return_pct'] = self.trades_df['return_pct'].mean()
            metrics['avg_win_return_pct'] = self.trades_df[self.trades_df['net_pnl'] > 0]['return_pct'].mean()
            metrics['avg_loss_return_pct'] = self.trades_df[self.trades_df['net_pnl'] < 0]['return_pct'].mean()

        if 'duration_days' in self.trades_df.columns:
            metrics['avg_duration_days'] = self.trades_df['duration_days'].mean()

        # Regime performance
        if 'regime' in self.trades_df.columns:
            for regime in self.trades_df['regime'].unique():
                regime_trades = self.trades_df[self.trades_df['regime'] == regime]
                if len(regime_trades) > 0:
                    metrics[f'{regime}_win_rate'] = (regime_trades['net_pnl'] > 0).mean()
                    metrics[f'{regime}_avg_return'] = regime_trades['return_pct'].mean() if 'return_pct' in regime_trades.columns else 0

        return metrics

    def plot_performance(self, df):
        """Create comprehensive performance plots"""
        if self.trades_df.empty:
            print("⚠ No trades to plot")
            return

        fig, axes = plt.subplots(3, 2, figsize=(16, 12))

        # 1. GBP/USD price with trades
        axes[0, 0].plot(df.index, df['GBP_USD'], 'k-', alpha=0.7, linewidth=1)

        # Plot trades
        long_trades = self.trades_df[self.trades_df['direction'] == 'long']
        short_trades = self.trades_df[self.trades_df['direction'] == 'short']

        if not long_trades.empty:
            axes[0, 0].scatter(long_trades['entry_date'], long_trades['entry_price'],
                              color='green', marker='^', s=50, alpha=0.7, label='Long')

        if not short_trades.empty:
            axes[0, 0].scatter(short_trades['entry_date'], short_trades['entry_price'],
                              color='red', marker='v', s=50, alpha=0.7, label='Short')

        axes[0, 0].set_title('GBP/USD Price with Trade Entries', fontweight='bold')
        axes[0, 0].set_ylabel('Price')
        axes[0, 0].legend()
        axes[0, 0].grid(alpha=0.3)

        # 2. Equity curve
        axes[0, 1].plot(self.strategy.equity_curve, 'b-', linewidth=2)
        axes[0, 1].axhline(y=self.strategy.initial_capital, color='r', linestyle='--', alpha=0.5)
        axes[0, 1].set_title('Equity Curve', fontweight='bold')
        axes[0, 1].set_ylabel('Capital ($)')
        axes[0, 1].grid(alpha=0.3)

        # 3. Drawdown
        axes[1, 0].plot(self.strategy.equity_curve, 'b-', alpha=0.3)
        peak = np.maximum.accumulate(self.strategy.equity_curve)
        drawdown = (np.array(self.strategy.equity_curve) - peak) / peak * 100
        axes[1, 0].fill_between(range(len(drawdown)), drawdown, 0, alpha=0.3, color='red')
        axes[1, 0].set_title('Drawdown', fontweight='bold')
        axes[1, 0].set_ylabel('Drawdown (%)')
        axes[1, 0].grid(alpha=0.3)

        # 4. Trade returns distribution
        if 'return_pct' in self.trades_df.columns:
            axes[1, 1].hist(self.trades_df['return_pct'], bins=20, alpha=0.7, color='steelblue')
            axes[1, 1].axvline(x=0, color='red', linestyle='--', alpha=0.5)
            axes[1, 1].set_title('Trade Returns Distribution', fontweight='bold')
            axes[1, 1].set_xlabel('Return (%)')
            axes[1, 1].set_ylabel('Frequency')
            axes[1, 1].grid(alpha=0.3)

        # 5. Regime performance
        if 'regime' in self.trades_df.columns:
            regime_stats = self.trades_df.groupby('regime').agg({
                'return_pct': 'mean',
                'net_pnl': lambda x: (x > 0).mean()
            }).round(3)

            if not regime_stats.empty:
                x = range(len(regime_stats))
                axes[2, 0].bar(x, regime_stats['return_pct'], alpha=0.7)
                axes[2, 0].set_xticks(x)
                axes[2, 0].set_xticklabels(regime_stats.index, rotation=45)
                axes[2, 0].set_title('Average Return by Regime', fontweight='bold')
                axes[2, 0].set_ylabel('Avg Return (%)')
                axes[2, 0].grid(alpha=0.3)

        # 6. Performance metrics table
        axes[2, 1].axis('off')
        metrics = self.calculate_metrics()

        table_data = [
            ['Total Return', f"{metrics.get('total_return_pct', 0):.2f}%"],
            ['Total Trades', f"{metrics.get('total_trades', 0)}"],
            ['Win Rate', f"{metrics.get('win_rate', 0):.1%}"],
            ['Sharpe Ratio', f"{metrics.get('sharpe_ratio', 0):.2f}"],
            ['Max Drawdown', f"{metrics.get('max_drawdown_pct', 0):.1f}%"],
            ['Profit Factor', f"{metrics.get('profit_factor', 0):.2f}"],
        ]

        table = axes[2, 1].table(cellText=table_data,
                                cellLoc='center',
                                loc='center',
                                colWidths=[0.4, 0.4])
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 1.5)

        plt.tight_layout()
        plt.show()

# =============================================================================
# 7. MAIN EXECUTION - COMPLETE
# =============================================================================

def main():
    print("="*100)
    print("🏴󠁧󠁢󠁥󠁮󠁧󠁿 ULTIMATE GBP/USD AUTOARIMA MEAN REVERSION STRATEGY")
    print("="*100)

    # API keys (REPLACE WITH YOUR ACTUAL KEYS)
    FRED_API_KEY = '3d986ceb3d77453d22b3955a5289e0b5'  # Your FRED key
    OANDA_TOKEN = '0ad694b953160d210907f7f268858831-e3c6206149c72264c10d8f9ab15f55d0'  # Your OANDA token

    print(f"\n🔑 Using:")
    print(f"   FRED API Key: {FRED_API_KEY[:10]}...")
    print(f"   OANDA Token: {OANDA_TOKEN[:10]}...")

    # Load data
    df = load_gbp_data_with_regimes(FRED_API_KEY, OANDA_TOKEN)

    if df is None:
        print("\n❌ Failed to load data. Please check:")
        print("   1. Your OANDA token is valid and has GBP/USD permissions")
        print("   2. Your FRED API key is valid")
        print("   3. Internet connection is working")
        print("   4. Date range is valid (2014-01-01 to 2026-01-21)")
        return

    print(f"\n📊 Data Summary:")
    print(f"   Total periods: {len(df)}")
    print(f"   Date range: {df.index.min()} to {df.index.max()}")
    print(f"   GBP/USD stats - Mean: {df['GBP_USD'].mean():.4f}, Std: {df['GBP_USD'].std():.4f}")
    print(f"   Available columns: {len(df.columns)}")

    # Split data for backtesting
    # split_date = pd.Timestamp('2023-12-01')
    split_date = pd.Timestamp('2023-12-01', tz='UTC')
    train_data = df[df.index <= split_date]

    print(f"\n🔧 Strategy Configuration:")
    print(f"   Training period: {len(train_data)} periods")
    print(f"   Initial capital: ${100000:,.0f}")
    print(f"   Strategy parameters loaded")

    # Initialize and run strategy
    strategy = UltimateGBPStrategy(initial_capital=100000)
    strategy.run_backtest(train_data, verbose=True)

    # Analyze performance
    print(f"\n📈 Performance Analysis:")
    print("="*50)

    analytics = GBPPerformanceAnalytics(strategy)
    metrics = analytics.calculate_metrics()

    if metrics['total_trades'] > 0:
        print(f"\n🎯 Core Metrics:")
        print(f"   Total Return: {metrics['total_return_pct']:.2f}%")
        print(f"   Total Trades: {metrics['total_trades']}")
        print(f"   Win Rate: {metrics['win_rate']:.1%}")
        print(f"   Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
        print(f"   Max Drawdown: {metrics['max_drawdown_pct']:.1f}%")
        print(f"   Profit Factor: {metrics['profit_factor']:.2f}")

        print(f"\n📊 Trade Statistics:")
        print(f"   Avg Return per Trade: {metrics.get('avg_return_pct', 0):.2f}%")
        print(f"   Avg Win: {metrics.get('avg_win_return_pct', 0):.2f}%")
        print(f"   Avg Loss: {metrics.get('avg_loss_return_pct', 0):.2f}%")
        print(f"   Avg Duration: {metrics.get('avg_duration_days', 0):.1f} days")

        # Regime performance
        print(f"\n🏴󠁧󠁢󠁥󠁮󠁧󠁿 Regime Performance:")
        for key in metrics:
            if key.endswith('_win_rate') and metrics[key] > 0:
                regime = key.replace('_win_rate', '')
                print(f"   {regime}: Win Rate: {metrics[key]:.1%}, Avg Return: {metrics.get(f'{regime}_avg_return', 0):.2f}%")

        # Plot results
        print(f"\n📈 Generating performance charts...")
        analytics.plot_performance(train_data)

    else:
        print(f"\n⚠ No trades executed")
        print(f"   Possible reasons:")
        print(f"   1. Signal quality threshold too high ({strategy.params['min_signal_quality']})")
        print(f"   2. Insufficient price deviation from forecasts")
        print(f"   3. Regime restrictions prevented trading")
        print(f"   4. Position sizing constraints too tight")

    print(f"\n" + "="*100)
    print(f"✅ STRATEGY EXECUTION COMPLETE")
    print("="*100)
    print(f"\n📋 COMPLETE IMPLEMENTATION STATUS:")
    print(f"   1. ✅ OANDA data loading with real OHLC")
    print(f"   2. ✅ FRED macro data integration")
    print(f"   3. ✅ Complete feature engineering")
    print(f"   4. ✅ Regime detection and classification")
    print(f"   5. ✅ AutoARIMA forecasting")
    print(f"   6. ✅ Position sizing with risk management")
    print(f"   7. ✅ Trade execution with stops and targets")
    print(f"   8. ✅ Performance analytics and visualization")
    print(f"   9. ✅ Real GBP/USD data pipeline (NO synthetic data)")

if __name__ == "__main__":
    main()

🏴󠁧󠁢󠁥󠁮󠁧󠁿 ULTIMATE GBP/USD AUTOARIMA MEAN REVERSION STRATEGY

🔑 Using:
   FRED API Key: 3d986ceb3d...
   OANDA Token: 0ad694b953...
🔄 Loading GBP/USD OHLC from OANDA...
✅ Received 3132 candles from OANDA
✅ Processed 3132 OHLC records
   Date range: 2014-01-01 22:00:00+00:00 to 2026-01-20 22:00:00+00:00
   GBP/USD range: 1.0686 - 1.7166

🔄 Loading macro data from FRED...
  ✓ UK_PROD: GBRPROCONAISMEI (69 points)
  ✓ US_PROD: OPHNFB (315 points)
  ✓ UK_RATE: IUDSOIA (7580 points)
  ✓ US_RATE: FEDFUNDS (858 points)
  ✓ UK_INFLATION: GBRCPIALLQINMEI (281 points)
  ✓ US_INFLATION: CPIAUCSL (948 points)
  ✓ VIX: VIXCLS (9407 points)
✅ Loaded 7 macro series

🔄 Merging OANDA + FRED data...
✅ Final dataset shape: (3132, 12)
   Columns: ['GBP_USD', 'GBP_USD_HIGH', 'GBP_USD_LOW', 'GBP_USD_OPEN', 'GBP_USD_VOLUME', 'UK_PROD', 'US_PROD', 'UK_RATE', 'US_RATE', 'UK_INFLATION', 'US_INFLATION', 'VIX']

🔄 Applying feature engineering...
✅ Feature engineering complete
   Final data shape: (3132, 35)

📊 Data 